# **Aula : Problemas de Interesse**

## Introdução

## **Allen-Cahn**

### **Equação Governante** 

A equação governante adotada segue a forma:
  $$\frac{\partial u}{\partial t} = \mu \left( \epsilon^2 \frac{\partial^2 u}{\partial x^2} - f(u) \right)$$

---

### **Fluxo Real**
Existem duas configurações comuns para o contexto de Allen-Cahn.

Na primeira o termo não-linear $f(u)$ no código é parametrizado como:
$$f(u) = -\beta_{fe} u$$

Na segunda o potencial duplo (cúbico), que é o mais tradicional para o problema, é definido por:
$$f(u) = u^3 - u$$

---

### **Condições Iniciais (IC)** 

* **Analítica:** O sistema é inicializado no domínio $x \in [0, 4\pi]$ com um perfil cossenoidal:
$$u(x,0) = \cos(0.5x)$$

* **Periódica:** O estado inicial no domínio $x \in [0, 2\pi]$ é uma onda senoidal transladada verticalmente:
$$u(x,0) = 0.8 + \sin(x)$$

---

### **Condições de Contorno (BC)** 

No contexto do DG a condição de contorno pode ser feita via *Ghost Cells*, impondo a forma fraca por meio do fluxo numérico na interface da borda. Para tal fluxo, calcula-se o estado numérico do traço (a média entre o elemento interno $u_{int}$ e o estado fantasma externo $u_{ghost}$):

$$u_{borda}^* = \frac{u_{int} + u_{ghost}}{2}$$

Se queremos que o valor físico efetivamente aplicado na borda seja o valor prescrito $g(t)$, igualamos o traço a $g(t)$:
$$\frac{u_{int} + u_{ghost}}{2} = g(t) \implies u_{ghost} = -u_{int} + 2g(t)$$

```Python
#Recovers the physical solution uh at the left boundary of the first element and at
#the right boundary of the last element (using the first row of matrices Flkp1 and Frk)
uhL = np.dot(Flkp1[0,:],ut[:,1,0])
uhR = np.dot(Frk[0,:],ut[:,-2,0])
```

* `uhL` representa $u_{int}$ (o valor da solução interpolado na borda interna esquerda).
* `uhR` representa $u_{int}$ (o valor da solução interpolado na borda interna direita).

#### **Análitica (Dirichlet Não Homogêneo)**
* Física: É aplicada uma condição de Dirichlet dependente do tempo baseada na evolução da solução analítica exata nas bordas:

$$u(0,t) = e^{-t} \cos(0.5x_{min})$$
$$u(L,t) = e^{-t} \cos(0.5x_{max})$$

```Python
# Analytical (Homogeneous Dirichlet)
ut[0,0,0] = -uhL + 2.0*(np.exp(-t)*np.cos(0.5*xmin)) #entrance
ut[0,-1,0] = -uhR + 2.0*(np.exp(-t)*np.cos(0.5*xmax)) #exit
```

#### **Dirichlet Não Homogêneo (Estático)** 
* Física: Fixa os valores das bordas no tempo $t$ com os mesmos valores da Condição Inicial em $t=0$.
* Contexto: Usado quando se quer simular um domínio onde os reservatórios nas extremidades são mantidos com concentrações/fases constantes.

```Python
# Non homogeneous Dirichlet
ut[0,0,0] = -uhL + 2.0*(0.8 + np.sin(xc[0,0]))    #entrance (Dirichlet)
ut[0,-1,0] = -uhR + 2.0*(0.8 + np.sin(xc[0,0]))    #exit (Dirichlet)
```

#### **Dirichlet Homogêneo**
* Física: $u(0,t) = 0$ e $u(L,t) = 0$.
* Contexto: Representa o estado de fase neutro $u=0$ nas paredes/fronteiras.

```Python
# Homogeneous Dirichlet
ut[0,0,0] = -uhL    #entrance (Dirichlet)
ut[0,-1,0] = -uhR    #exit (Dirichlet)
```

#### **Neumann Homogêneo**

* Física: o estado adjacente da borda copia o estado do elemento:

$$\left. \frac{\partial u}{\partial x} \right |_{x=0} = 0 \quad \text{e} \quad \left. \frac{\partial u}{\partial x} \right |_{x=L} = 0$$

* Contexto: Como $u_{ghost} = u_{int}$, a diferença na interface é nula ($[[u]] = 0$). Esta é a condição de contorno mais clássica e fisicamente natural para a Equação de Allen-Cahn, pois representa um sistema isolado (fronteiras impermeáveis), onde a dinâmica de separação de fases ocorre sem troca de massa com o meio externo. 

```Python
# Homogeneous Neumann at left and right boundaries
ut[0,0,0] = uhL     #entrance 
ut[0,-1,0] = uhR    #exit
```

#### **Periódica**
* Física: Simula um domínio infinito ou "em anel", onde o fluxo que sai pela direita entra pela esquerda ($u(0,t) = u(L,t)$).
* Contexto: Muito útil em Allen-Cahn para estudar o crescimento e aniquilação de domínios (interfaces de fase) longe de qualquer efeito de parede/fronteira. 

```Python
# Periodic BC
ut[0,0,0] = -uhL + 2.0*(0.8 + np.sin(xc[0,0])) #entrance
ut[0,-1,0] = ut[0,0,0]  #exit
```

---
 

## **Cahn-Hillard**

### **Equação Governante**

Aqui temos um sistema acoplado Cahn-Hilliard / Allen-Cahn, lidando com dois parâmetros de ordem: $u$ (fase conservada, regida por Cahn-Hilliard) e $v$ (fase não-conservada, regida por Allen-Cahn).  As equações governantes adotadas seguem a a forma:

$$\frac{\partial u}{\partial t} = \frac{\partial}{\partial x} \left[ b(u,v) \frac{\partial}{\partial x} \left( \frac{\partial \Psi}{\partial u} - \gamma \frac{\partial^2 u}{\partial x^2} \right) \right]$$

$$\frac{\partial v}{\partial t} = - \rho^{-1} b(u,v) \left( \frac{\partial \Psi}{\partial v} - \gamma \frac{\partial^2 v}{\partial x^2} \right)$$

---

### **Fluxo Real**
O fluxo e as não-linearidades deste sistema são divididos em duas partes: a função de mobilidade $b(u,v)$ e as derivadas da Energia Livre $\Psi(u,v)$. 

#### Função de Mobilidade $b(u,v)$:
Na primeira configuração (geralmente usada com condições flutuantes), a mobilidade é dependente do estado:
$$b(u,v) = u(1-u)(0.25 - v^2)$$

Na segunda configuração (usada para validação analítica), a mobilidade é constante:
$$b(u,v) = 1.0$$

#### Derivadas da Energia Livre $\Psi(u,v)$:

A não-linearidade termodinâmica inclui interações entrópicas (termos logarítmicos) e entálpicas:

$$\frac{\partial \Psi}{\partial u} = \Theta \left[ \ln\left(\frac{u+v}{1-u-v}\right) + \ln\left(\frac{u-v}{1-u+v}\right) \right] + \frac{\alpha}{2}(1-2u)$$

$$\frac{\partial \Psi}{\partial v} = \Theta \left[ \ln\left(\frac{u+v}{1-u-v}\right) - \ln\left(\frac{u-v}{1-u+v}\right) \right] - \beta v$$

Uma versão simplificada com o termo $\Theta$ desligado para testes puramente analíticos também pode ser usada:

$$\frac{\partial \Psi}{\partial u} = \frac{\alpha}{2}(1-2u)$$

$$\frac{\partial \Psi}{\partial v} = - \beta v$$

---

### **Condições Iniciais**
* **Analítica:** O sistema é inicializado para evoluir de acordo com a solução exata esperada:
$$u(x,0) = \cos(0.5x) - \sin(x)$$
$$v(x,0) = \cos(0.5x)$$

* **Manufaturada:** Outra configuração suave para testes de convergência:
$$u(x,0) = 0.5 - e^{-2.0}\sin(x)$$
$$v(x,0) = e^{-4.0}\cos(0.5x)$$

* **Flutuante (Determinística):** Uma perturbação senoidal com decaimento exponencial, usada para iniciar a separação de fases:
    * Para $u(x,0)$ existem três configurações possíveis
            $$u(x,0) = 0.55 - e^{-3.0}\sin(6\pi x)$$
            $$u(x,0) = 0.55 - e^{-3.0}\sin(6\pi x + \frac{10}{21}\pi)$$
            $$u(x,0) = 0.55 - e^{-3.0}\sin(6\pi x - \frac{10}{21}\pi)$$
    * Para $v(x,0)$ existem duas configurações possíveis   
            $$v(x,0) = e^{-4.0}\cos(2\pi x)$$
            $$v(x,0) = 0.01$$            

* **Flutuante (Estocástica / Arquivo):** Inserção de ruído aleatório em $u$ e/ou $v$ e a leitura de um perfil prévio para $u$ e/ou $v$:
  * Ruído Aleatório:
    $$u(x,0) = \text{Uniforme}(-0.05, 0.05)$$
    $$v(x,0) = \text{Uniforme}(-0.002, 0.002)$$
  * Com arquivo prévio
    $$u(x,0) = \text{Carregado de arquivo (icr\_u.npy)}$$
    $$v(x,0) = \text{Carregado de arquivo (icr\_v.npy)}$$


---

### **Condições de Contorno**
Assim como explicado no Allen-Cahn se queremos que o valor físico efetivamente aplicado na borda seja o valor prescrito $g(t)$
$$u_{ghost} = -u_{int} + 2g(t)$$

```Python
#Recovers the physical solution uh at the left boundary of the first element and at
#the right boundary of the last element (using the first row of matrices Flkp1 and Frk)
uhR = np.dot(Frk[0,:],ut[:,-2,0])
vhR = np.dot(Frk[0,:],vt[:,-2,0])
uhL = np.dot(Flkp1[0,:],ut[:,1,0])
vhL = np.dot(Flkp1[0,:],vt[:,1,0])
```

#### **Analítica (Dirichlet Não Homogêneo)**
* Física: É aplicada uma condição de Dirichlet dependente do tempo, baseada na evolução temporal forçada nas bordas para ambas as variáveis.  
* Contexto: Usado com o Método das Soluções Manufaturadas (MMS) para validar a precisão e a estabilidade do solver.

```Python
# Non homogeneous Dirichlet
ut[0,-1,0] = -uhR + 2.0*(np.exp(t)*np.cos(0.5*xmax) - np.exp(-0.5*t)*np.sin(xmax)) #exit
vt[0,-1,0] = -vhR + 2.0*(np.exp(-t)*np.cos(0.5*xmax)) #exit
ut[0,0,0] = -uhL + 2.0*(np.exp(t)*np.cos(0.5*xmin) - np.exp(-0.5*t)*np.sin(xmin)) #entrance
vt[0,0,0] = -vhL + 2.0*(np.exp(-t)*np.cos(0.5*xmin)) #entrance
```

#### **Dirichlet Homogêneo**
* Física: Fixa o valor das variáveis em zero nas extremidades do domínio ($u=0, v=0$).  
* Contexto: Representa o ancoramento de uma fase específica (fase neutra) nas paredes do sistema físico.

``` Python
# Homogeneous Dirichlet
ut[0,0,0] = -uhL    #entrance (Dirichlet)
ut[0,-1,0] = -uhR   #exit (Dirichlet)
```

#### **Neumann Homogêneo**
* Física: O estado adjacente da borda copia o estado do elemento, forçando gradientes nulos nas fronteiras para ambos os parâmetros:
$$\left. \frac{\partial u}{\partial x} \right |_{borda} = 0 \quad \text{e} \quad \left. \frac{\partial v}{\partial x} \right |_{borda} = 0$$

* Contexto: Esta é a condição física mais importante para Cahn-Hilliard, pois impõe fluxo de massa nulo. Fronteiras impermeáveis garantem que a massa total do parâmetro conservado $u$ permaneça constante durante a separação de fases.

```Python
# Homogeneous Neumann at left and right boundaries
ut[0,-1,0] = uhR
vt[0,-1,0] = vhR
```

> Nota 1: O mesmo tratamento é matematicamente replicado ou estendido para a borda esquerda e para as variáveis auxiliares do sistema acoplado, como os fluxos $q_1$ e $r_1$

> Nota 2: Existe o uso de **duas projeções de fluxo** no Cahn-Hillard, a *MobilityProjection* e a *FreeEnergyProjection*

---

## **Euler**

### **Equação Governante**

### **Fluxo Real**

### **Condições Iniciais**

### **Condições de Contorno**

## **Burgers Inviscido**

### **Equação Governante**

### **Fluxo Real**

### **Condições Iniciais**

### **Condições de Contorno**

## **Advecção Linear**

### **Equação Governante**

### **Fluxo Real**

### **Condições Iniciais**

### **Condições de Contorno**

## **Shallow-Water**

### **Equação Governante**

### **Fluxo Real**

### **Condições Iniciais**

### **Condições de Contorno**

## **Shock-Density**

### **Equação Governante**

### **Fluxo Real**

### **Condições Iniciais**

### **Condições de Contorno**

## **Shu-Osher**

### **Equação Governante**

### **Fluxo Real**

### **Condições Iniciais**

### **Condições de Contorno**

## **Traffic-Flow**

### **Equação Governante**

### **Fluxo Real**

### **Condições Iniciais**

### **Condições de Contorno**

## **Burgers Viscoso**

### **Equação Governante**

### **Fluxo Real**

### **Condições Iniciais**

### **Condições de Contorno**